# P79 — Bosques aleatorios

## 1. Título y paper

**Paper:** *Random Forests*  
**Autoría:** Leo Breiman  
**Año y venue:** 2001 · Machine Learning, 45(1), 5–32  
**Nivel:** L3 · **Motor:** `random_forest`  
**Ficha completa:** [`P79_random_forest`](../../papers/foundational/P79_random_forest/README.md)

**Hito:** Demuestra que el error de un conjunto depende de la fuerza de sus miembros Y de su correlación, y que empeorarlos a propósito puede mejorarlo.

- [doi:10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El bagging reducía la varianza promediando árboles entrenados sobre remuestreos, pero los árboles seguían pareciéndose demasiado: ante los mismos datos elegían casi siempre las mismas variables.
2. Ejecutar una implementación mínima de la propuesta: Añadir una segunda fuente de azar: en cada nodo, considerar solo un subconjunto aleatorio de variables. Los árboles empeoran individualmente y se descorrelacionan, y la cota del error del bosque mejora.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Breiman (1996), bagging
- Ho (1998), subespacios aleatorios
- P74


## 4. Intuición

Si promedias cien opiniones idénticas, obtienes la misma opinión. Para que promediar sirva, las opiniones tienen que ser distintas. Breiman fuerza esa diferencia limitando a propósito lo que cada árbol puede mirar — y sus árboles empeoran, y el bosque mejora.


## 5. Concepto mínimo

```text
Error del bosque ≲ ρ̄·(1 − s²)/s²

    ρ̄ = correlación media entre árboles     ← bajar esto es la aportación
    s  = fuerza media de cada árbol         ← subir esto es lo obvio

bagging          → diversidad por los DATOS
subespacio de m  → diversidad por las PREGUNTAS que cada árbol puede hacer
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('random_forest', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Mejora el bosque a su árbol medio en todos los casos?
2. ¿Qué le pasa al acuerdo entre árboles al bajar las variables por árbol?
3. ¿Y al árbol individual?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('random_forest', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('random_forest', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El bosque mejora a su árbol medio en las cuatro configuraciones. Al bajar de 8 variables por árbol a 2, el **acuerdo cae de 0,7238 a 0,5399** —los árboles se descorrelacionan— y el **árbol medio empeora de 0,3087 a 0,4047**. Las dos cosas se mueven a la vez, en direcciones opuestas.


## 10. Comentario pedagógico

Por eso `m` es un hiperparámetro y no una constante: arbitra entre fuerza y correlación, y su óptimo depende de los datos. En esta tabla gana `m = 8`; en otros conjuntos gana un valor pequeño. Lo transferible no es el número: es que existe el compromiso y hay que buscarlo.


## 11. Error o anti-patrón deliberado

Anti-patrón: creer que un bosque mejora siempre por tener más árboles.


In [ ]:
print('Anadir arboles reduce la varianza del voto y satura pronto.')
print('Lo que NO hace es reducir el sesgo: si todos los arboles se equivocan igual,')
print('promediar mil de ellos devuelve el mismo error.')

## 12. Corrección

Lo que sí mueve la aguja, con el barrido delante:


In [ ]:
r = run_paper_lab('random_forest', seed=7)['result']
for b in r['barrido_de_variables_por_arbol']:
    print(f"m={b['variables_por_arbol']:>2}  arbol medio={b['error_medio_de_un_arbol']:<8}"
          f" bosque={b['error_del_bosque']:<8} acuerdo={b['acuerdo_medio_entre_arboles']}")
print('mejor configuracion aqui:', r['mejor_configuracion'])

## 13. Desafío guiado

Comprueba en la salida que el acuerdo baja de forma monótona al reducir `m`, y que el error del árbol individual sube de forma monótona. Explica por qué eso implica que existe un óptimo.


In [ ]:
r = run_paper_lab('random_forest', seed=3)['result']
show(r)

## 14. Desafío autónomo

Entrena un bosque real sobre un conjunto tabular, barre `m` y dibuja las dos curvas: error del árbol medio y error del bosque. Localiza el óptimo y compáralo con la heurística `√p`.


## 15. Evidencia de aprendizaje

Guarda el barrido con las tres columnas y tu explicación del compromiso entre fuerza y correlación.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P79_random_forest/README.md) · evaluación formal: [`assessments/papers/P79_random_forest.md`](../../assessments/papers/P79_random_forest.md)


## 16. Cierre

Dos artículos del mismo autor y del mismo año. El segundo no propone un método: propone una discusión sobre para qué sirven los modelos.


## 17. Conexión con el siguiente hito

- P80

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
